
Limpieza del dataset - Encuesta Distrital de Percepción y Cultura Ciudadana
Eje: Violencia contra la mujer
DataJam - filtrado y limpieza inicial


In [60]:
import pandas as pd
import seaborn as sb
import numpy as np
import os
import csv
import io
import matplotlib.pyplot as plt

# 1. CONFIGURACIÓN


In [61]:
RUTA_ENTRADA = '../data/base_ano_movil_2025.csv'

COLUMNAS_VIOLENCIA_MUJER = [
    # Llaves / identificación
    "DIRECTORIO_MZ", "DIRECTORIO_PRED", "DIRECTORIO_HOG", "DIRECTORIO_PER",

    # Llaves geográficas y temporales (cruce)
    "periodo", "SECTOR", "Cod_Locali", "Nom_Locali", "Cod_UPL", "Nom_UPL",

    # Perfil sociodemográfico
    "A6x2", "A6x3", "C1", "D1", "E1", "E1x1", "G1", "H1",
    "C303", "sexo_jefe",

    # Roles de género / distribución de tareas domésticas
    "Ax201", "Bx201", "Cx201", "Dx201", "Ex201", "Fx201",
    "Gx201", "Hx201", "Ix201", "Jx201",
    "Ax202", "ind_distribuciontareas_202",

    # Violencia intrafamiliar (delito sufrido)
    "Jx402", "Jx403",

    # Acoso sexual, violencia intrafamiliar y contra la mujer presenciados (por lugar)
    "Kx404_1", "Kx404_2", "Kx404_3", "Kx404_4", "Kx404_5", "Kx404_6",
    "Lx404_1", "Lx404_2", "Lx404_3", "Lx404_4", "Lx404_5", "Lx404_6",
    "Mx404_1", "Mx404_2", "Mx404_3", "Mx404_4", "Mx404_5", "Mx404_6",
    "Nx404_1", "Nx404_2", "Nx404_3", "Nx404_4", "Nx404_5", "Nx404_6",

    # Percepción de seguridad
    "F405", "G406", "IPS_dia", "IPS_noche",
    "IPSJ_A", "IPSJ_C", "IPSJ_E",

    # Inclusión, diversidad, confianza
    "Dx704", "ICG_D", "Bx704", "ICG_B",

    # Salud mental (GAD-7)
    "Ax102", "Bx102", "Cx102", "Dx102", "Ex102", "Fx102", "Gx102",
    "A101", "ind_salud_101", "ind_salud_102",

    # Factores de expansión (para estimaciones representativas)
    "fexp_calp_anu", "fexp_calh_anu",
]


# 2. CARGA DEL DATASET

In [62]:
print("Cargando dataset...")
df = pd.read_csv(RUTA_ENTRADA, encoding="utf-8", low_memory=False)

# Elimina columna índice sin nombre si viene del export original (";","")
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]

print(f"Dimensiones originales: {df.shape}")

Cargando dataset...
Dimensiones originales: (13082, 286)


# 3. VALIDAR QUE LAS COLUMNAS EXISTEN

In [63]:
faltantes = [c for c in COLUMNAS_VIOLENCIA_MUJER if c not in df.columns]
if faltantes:
    print("Advertencia: columnas no encontradas en el dataset:")
    print(faltantes)

columnas_disponibles = [c for c in COLUMNAS_VIOLENCIA_MUJER if c in df.columns]

# 4. FILTRAR COLUMNAS DE INTERÉS


In [64]:
df_filtrado = df[columnas_disponibles].copy()
print(f"Dimensiones tras filtrar columnas: {df_filtrado.shape}")

Dimensiones tras filtrar columnas: (13082, 81)


# 5. LIMPIEZA DE VALORES


In [65]:
# 5.1 Normalizar strings "NA", "N/A", "" a NaN reales
df_filtrado = df_filtrado.replace(
    to_replace=["NA", "N/A", "na", "n/a", "", "-", "999", "9999"],
    value=np.nan
)

# 5.2 Convertir columnas binarias (0/1) de presenciados (K,L,M,N x404) a numérico
cols_binarias = [
    c for c in columnas_disponibles
    if c.startswith(("Kx404", "Lx404", "Mx404", "Nx404"))
]
for col in cols_binarias:
    df_filtrado[col] = pd.to_numeric(df_filtrado[col], errors="coerce")

# 5.3 Convertir columnas de escala/indicadores a numérico
cols_indicadores = [
    c for c in columnas_disponibles
    if c.startswith(("IPS_", "IPSJ_", "ICG_", "ind_salud", "ind_distribuciontareas"))
    or c in ["F405", "G406", "A101"]
]
for col in cols_indicadores:
    df_filtrado[col] = pd.to_numeric(df_filtrado[col], errors="coerce")

# 5.4 Normalizar variables categóricas de sexo/género
if "D1" in df_filtrado.columns:
    df_filtrado["D1"] = df_filtrado["D1"].astype("Int64")

if "sexo_jefe" in df_filtrado.columns:
    df_filtrado["sexo_jefe"] = df_filtrado["sexo_jefe"].astype("Int64")

# 5.5 Normalizar periodo a formato consistente (año-mes)
if "periodo" in df_filtrado.columns:
    df_filtrado["periodo"] = df_filtrado["periodo"].astype(str).str.strip()

# 5.6 Normalizar nombres de localidad (mayúsculas/espacios)
if "Nom_Locali" in df_filtrado.columns:
    df_filtrado["Nom_Locali"] = (
        df_filtrado["Nom_Locali"].astype(str).str.strip().str.upper()
    )

# 6. DESCARGAR EL CSV

In [66]:
# Ruta relativa desde scripts/ hacia outputs/
RUTA_SALIDA = os.path.join("..", "outputs", "base_violencia_mujer_limpia.csv")

df_filtrado.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"Dataset guardado en: {os.path.abspath(RUTA_SALIDA)}")

Dataset guardado en: c:\Users\tralf\Documents\u_distrital\datajam\DataJamDistrital_Bogota2026\outputs\base_violencia_mujer_limpia.csv


In [67]:
ruta = '../outputs/base_violencia_mujer_limpia.csv'
ruta2 = '../outputs/delitossexuales.csv'

df = pd.read_csv(ruta)
df.describe(include='all').T

df2 = pd.read_csv(ruta2)
df2.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Localidad,80,20,Antonio Nariño,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Total,80.0,NaN,NaN,NaN,131.575,118.015422,0.0,34.0,107.0,192.5,421.0
Porcentaje,80.0,NaN,NaN,NaN,2.626125,2.405158,0.0,0.0375,2.2,4.025,8.2
PobMujeres,80.0,NaN,NaN,NaN,206779.1875,183676.912385,1788.0,69152.5,160788.5,317553.0,652069.0
Tasa,80.0,NaN,NaN,NaN,93.31425,97.080929,0.0,20.785,67.995,121.96,481.01
Fecha,80,4,2025-06-30,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
df2.describe(include='all')

,Localidad,Total,Porcentaje,PobMujeres,Tasa,Fecha
count,80,80.000000,80.000000,80.000000,80.000000,80
unique,20,NaN,NaN,NaN,NaN,4
top,Antonio Nariño,NaN,NaN,NaN,NaN,2025-06-30
freq,4,NaN,NaN,NaN,NaN,20
mean,NaN,131.575000,2.626125,206779.187500,93.314250,NaN
std,NaN,118.015422,2.405158,183676.912385,97.080929,NaN
min,NaN,0.000000,0.000000,1788.000000,0.000000,NaN
25%,NaN,34.000000,0.037500,69152.500000,20.785000,NaN
50%,NaN,107.000000,2.200000,160788.500000,67.995000,NaN
75%,NaN,192.500000,4.025000,317553.000000,121.960000,NaN


In [69]:
df2.dtypes

Localidad         str
Total         float64
Porcentaje    float64
PobMujeres      int64
Tasa          float64
Fecha             str
dtype: object

In [70]:
df2["Localidad"].unique().tolist()

['Antonio Nariño',
 'Tunjuelito',
 'Rafael Uribe Uribe',
 'La Candelaria',
 'Barrios Unidos',
 'Teusaquillo',
 'Puente Aranda',
 'Los Mártires',
 'Sumapaz',
 'Usaquén',
 'Chapinero',
 'Santa Fe',
 'San Cristóbal',
 'Usme',
 'Ciudad Bolívar',
 'Bosa',
 'Kennedy',
 'Fontibón',
 'Engativá',
 'Suba']

In [71]:
df["Nom_Locali"].unique().tolist()

['CIUDAD BOLIVAR',
 'USME',
 'BOSA',
 'SAN CRISTOBAL',
 'KENNEDY',
 'SUBA',
 'ENGATIVA',
 'SANTA FE',
 'FONTIBON',
 'TUNJUELITO',
 'RAFAEL URIBE URIBE',
 'ANTONIO NARIÑO',
 'TEUSAQUILLO',
 'CANDELARIA',
 'LOS MARTIRES',
 'CHAPINERO',
 'USAQUEN',
 'PUENTE ARANDA',
 'BARRIOS UNIDOS']

In [72]:
# Diccionario de mapeo: valor original -> nombre normalizado
MAPEO_LOCALIDADES = {
    "CIUDAD BOLIVAR": "Ciudad Bolívar",
    "USME": "Usme",
    "BOSA": "Bosa",
    "SAN CRISTOBAL": "San Cristóbal",
    "KENNEDY": "Kennedy",
    "SUBA": "Suba",
    "ENGATIVA": "Engativá",
    "SANTA FE": "Santa Fe",
    "FONTIBON": "Fontibón",
    "TUNJUELITO": "Tunjuelito",
    "RAFAEL URIBE URIBE": "Rafael Uribe Uribe",
    "ANTONIO NARIÑO": "Antonio Nariño",
    "TEUSAQUILLO": "Teusaquillo",
    "CANDELARIA": "La Candelaria",
    "LOS MARTIRES": "Los Mártires",
    "CHAPINERO": "Chapinero",
    "USAQUEN": "Usaquén",
    "PUENTE ARANDA": "Puente Aranda",
    "BARRIOS UNIDOS": "Barrios Unidos",
}

# Renombrar la columna
df = df.rename(columns={"Nom_Locali": "Localidad"})

# Aplicar el mapeo de valores
df["Localidad"] = df["Localidad"].map(MAPEO_LOCALIDADES)

# Verificación: valores que no hicieron match (deberían ser 0)
no_mapeados = df[df["Localidad"].isna() & df["Localidad"].notna()]
print("Valores sin mapear:", df["Localidad"].isna().sum())

# Verificar resultado
print(df["Localidad"].value_counts())

Valores sin mapear: 0
Localidad
Suba                  1802
Kennedy               1737
Bosa                  1250
Engativá              1232
Ciudad Bolívar         858
Usaquén                834
Usme                   744
San Cristóbal          732
Rafael Uribe Uribe     716
Fontibón               654
Puente Aranda          378
Teusaquillo            335
Tunjuelito             308
Santa Fe               300
Chapinero              296
Barrios Unidos         270
Antonio Nariño         246
Los Mártires           239
La Candelaria          151
Name: count, dtype: int64


In [73]:
df["periodo"].unique().tolist()

[202501,
 202502,
 202503,
 202504,
 202505,
 202506,
 202507,
 202508,
 202509,
 202510,
 202511,
 202512]

In [74]:
# Renombrar la columna
df = df.rename(columns={"periodo": "Fecha"})

# Convertir de formato YYYYMM (int) a string YYYY-MM-01
df["Fecha"] = pd.to_datetime(df["Fecha"].astype(str), format="%Y%m").dt.strftime("%Y-%m-01")

# Verificar resultado
df["Fecha"].unique().tolist()

['2025-01-01',
 '2025-02-01',
 '2025-03-01',
 '2025-04-01',
 '2025-05-01',
 '2025-06-01',
 '2025-07-01',
 '2025-08-01',
 '2025-09-01',
 '2025-10-01',
 '2025-11-01',
 '2025-12-01']

In [75]:
df["SECTOR"].unique().tolist()


['Sector Sur Oriente',
 'Sector Sur Occidente',
 'Sector Noroccidente',
 'Sector Occidente',
 'Sector Centro Ampliado',
 'Sector Norte']

In [76]:
df["Nom_UPL"].unique().tolist()

['Arborizadora',
 'Lucero',
 'Usme - Entrenubes',
 'Edén',
 'Suba',
 'Tibabuyes',
 'Engativá',
 'Centro Histórico',
 'Kennedy',
 'Britalia',
 'Fontibón',
 'Tintal',
 'Patio Bonito',
 'Porvenir',
 'Bosa',
 'Tunjuelito',
 'Rafael Uribe',
 'San Cristóbal',
 'Restrepo',
 'Teusaquillo',
 'Niza',
 'Chapinero',
 'Usaquén',
 'Toberín',
 'Rincón de Suba',
 'Tabora',
 'Salitre',
 'Puente Aranda',
 'Barrios Unidos',
 'Torca']

In [77]:
df = df.rename(columns={
    "SECTOR": "SectorUPL",
    "Nom_UPL": "Unidad_de_Planeamiento_Local_UPL"
})

In [78]:
df = df.drop(columns=[
    "DIRECTORIO_MZ",
    "DIRECTORIO_PRED",
    "DIRECTORIO_HOG",
    "DIRECTORIO_PER",
    "Cod_Locali",
    "Cod_UPL"
])

In [79]:
import os

# Ruta relativa desde scripts/ hacia outputs/
RUTA_SALIDA = os.path.join("..", "outputs", "Encuesta_percepcion_limpia.csv")

df.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"Dataset guardado en: {os.path.abspath(RUTA_SALIDA)}")

Dataset guardado en: c:\Users\tralf\Documents\u_distrital\datajam\DataJamDistrital_Bogota2026\outputs\Encuesta_percepcion_limpia.csv


In [80]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Fecha,13082,12,2025-04-01,1647,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SectorUPL,13082,6,Sector Sur Oriente,3035,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Localidad,13082,19,Suba,1802,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unidad_de_Planeamiento_Local_UPL,13082,30,Rafael Uribe,756,NaN,NaN,NaN,NaN,NaN,NaN,NaN
A6x2,13082.0,NaN,NaN,NaN,0.843755,1.725897,0.0,0.0,0.0,1.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...
A101,13082.0,NaN,NaN,NaN,3.799266,1.829789,1.0,3.0,4.0,4.0,99.0
ind_salud_101,13082.0,NaN,NaN,NaN,0.700887,0.457887,0.0,0.0,1.0,1.0,1.0
ind_salud_102,13082.0,NaN,NaN,NaN,0.274117,0.608719,0.0,0.0,0.0,0.0,3.0
fexp_calp_anu,13082.0,NaN,NaN,NaN,467.076135,451.054524,2.638768,192.424916,346.911851,593.985103,7490.925926
